In [1]:
import os
import textwrap
from dotenv import load_dotenv
from langchain_core.vectorstores import InMemoryVectorStore

from model.graph_rag import identify_matched_concepts, initialize_embedding_model, setup_graph_retriever
from model.text_processing import create_chunk_documents, split_paper_into_chunks
from model.ontology import load_ontology, build_ontology_graph, create_ontology_documents, calculate_ontology_depth
from utils.midas_api import get_paper_data
from model.scoring import calculate_confidence_scores, calculate_relevance_scores, calculate_terminology_scores
from model.reporting import analyze_chunk_similarities, print_top_matches, print_reasoned_paths, \
    print_gaps_and_next_steps, generate_verdict

/Users/harry/miniconda3/envs/ontgraphgrag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
api_key = os.getenv("MIDAS_API_KEY")
api_key

'9de0141353198384ca24ac41d2b38883'

In [3]:
paper_id = 1457 # has an paper abstract object with several strings in an array, instead of just one string
ontology_path = "midas-data.owl"

    # Get paper data
paper_data = get_paper_data(paper_id, api_key)
print("got paper_data")

print(f"# Paper Analysis Report\n")
print(f"**Paper ID:** {paper_id}")
print(f"**Title:** {paper_data['paper_title']}\n")
print(textwrap.fill(paper_data["paper_abstract"], width=80))
print(f"\n**Keywords:** {', '.join(paper_data['paper_keywords'])}")
print(f"**MeSH Terms:** {', '.join(paper_data['paper_meshterms'])}\n")

got paper_data
# Paper Analysis Report

**Paper ID:** 1457
**Title:** Forecasting national and regional influenza-like illness for the USA.

Health planners use forecasts of key metrics associated with influenza-like
illness (ILI); near-term weekly incidence, week of season onset, week of peak,
and intensity of peak. Here, we describe our participation in a weekly
prospective ILI forecasting challenge for the United States for the 2016-17
season and subsequent evaluation of our performance. We implemented a
metapopulation model framework with 32 model variants. Variants differed from
each other in their assumptions about: the force-of-infection (FOI); use of
uninformative priors; the use of discounted historical data for not-yet-observed
time points; and the treatment of regions as either independent or coupled.
Individual model variants were chosen subjectively as the basis for our weekly
forecasts; however, a subset of coupled models were only available part way
through the season. M

In [4]:
query = " ".join(paper_data['paper_keywords'] + paper_data['paper_meshterms']) + " "+paper_data['paper_title']
query

'Centers for Disease Control and Prevention, U.S. Computational Biology Epidemics Forecasting Humans Humidity Influenza, Human Markov Chains Models, Biological Models, Statistical Monte Carlo Method Prospective Studies Retrospective Studies Seasons United States Forecasting national and regional influenza-like illness for the USA.'

In [5]:
g, classes, properties = load_ontology(ontology_path)

Loaded 307 classes and 2 properties from the ontology.


In [6]:
g

<Graph identifier=N70892010deee40dd9af99a45ca709765 (<class 'rdflib.graph.Graph'>)>

In [7]:
G = build_ontology_graph(g, classes, properties)

In [8]:
G

In [9]:
# Create langchain document objects for ontology elements
ontology_docs, label_map = create_ontology_documents(G)

In [10]:
len(ontology_docs)

309

In [11]:
ontology_docs[0]

Document(id='http://w3id.org/midas-metadata/midas6', metadata={'type': 'Class', 'label': 'modeling', 'parents': ['http://w3id.org/midas-metadata/midas2'], 'children': ['http://w3id.org/midas-metadata/midas16', 'http://w3id.org/midas-metadata/midas97', 'http://w3id.org/midas-metadata/midas68', 'http://w3id.org/midas-metadata/midas15'], 'equivalentClasses': [], 'domainOf': [], 'rangeOf': [], 'synonyms': []}, page_content='modeling')

In [12]:
ontology_docs[1]

Document(id='http://purl.obolibrary.org/obo/MONDO_0004335', metadata={'type': 'Class', 'label': 'digestive system disorder', 'parents': ['http://purl.obolibrary.org/obo/DOID_4'], 'children': ['http://purl.obolibrary.org/obo/MONDO_0002251', 'http://purl.obolibrary.org/obo/MONDO_0000916', 'http://purl.obolibrary.org/obo/NCIT_C128351', 'http://purl.obolibrary.org/obo/MONDO_0001673'], 'equivalentClasses': [], 'domainOf': [], 'rangeOf': [], 'synonyms': []}, page_content='digestive system disorder')

In [13]:
chunks = split_paper_into_chunks(paper_data['paper_abstract'])

In [14]:
len(chunks)

2

In [15]:
chunks[0]

'Health planners use forecasts of key metrics associated with influenza-like illness (ILI); near-term weekly incidence, week of season onset, week of peak, and intensity of peak. Here, we describe our participation in a weekly prospective ILI forecasting challenge for the United States for the 2016-17 season and subsequent evaluation of our performance. We implemented a metapopulation model framework with 32 model variants. Variants differed from each other in their assumptions about: the force-of-infection (FOI); use of uninformative priors; the use of discounted historical data for not-yet-observed time points; and the treatment of regions as either independent or coupled. Individual model variants were chosen subjectively as the basis for our weekly forecasts; however, a subset of coupled models were only available part way through the season. Most frequently, during the 2016-17 season, we chose; FOI variants with both school vacations and humidity terms; uninformative priors; the i

In [16]:
chunks[1]

'discounted historical data for not-yet-observed time points; and coupled regions (when available). Our near-term weekly forecasts substantially over-estimated incidence early in the season when coupled models were not available. However, our forecast accuracy improved in absolute terms and relative to other teams once coupled solutions were available. In retrospective analysis, we found that the 2016-17 season was not typical: on average, coupled models performed better when fit without historically augmented data. Also, we tested a simple ensemble model for the 2016-17 season and found that it underperformed our subjective choice for all forecast targets. In this study, we were able to improve accuracy during a prospective forecasting exercise by coupling dynamics between regions. Although reduction of forecast subjectivity should be a long-term goal, some degree of human intervention is likely to improve forecast accuracy in the medium-term in parallel with the systematic considerat

how do chunks relate? What are in the resulting documents?  - just a split in two here.

In [17]:
paper_chunk_docs = create_chunk_documents(chunks, G)
len(paper_chunk_docs)

2

In [18]:
paper_chunk_docs[0]

Document(id='paper_chunk_0', metadata={'type': 'PaperChunk', 'mentions': ['http://w3id.org/midas-metadata/midas111', 'http://purl.obolibrary.org/obo/MONDO_0005812', 'http://w3id.org/midas-metadata/midas18']}, page_content='Health planners use forecasts of key metrics associated with influenza-like illness (ILI); near-term weekly incidence, week of season onset, week of peak, and intensity of peak. Here, we describe our participation in a weekly prospective ILI forecasting challenge for the United States for the 2016-17 season and subsequent evaluation of our performance. We implemented a metapopulation model framework with 32 model variants. Variants differed from each other in their assumptions about: the force-of-infection (FOI); use of uninformative priors; the use of discounted historical data for not-yet-observed time points; and the treatment of regions as either independent or coupled. Individual model variants were chosen subjectively as the basis for our weekly forecasts; howe

has 3 mentions. what are the terms that are mentions? 

- MONDO_0005812 is influenza - looks good
- midas111 is evaluation
 - midas18 is forecasting

should we be putting the text instead of the label in the mentions?


In [19]:
paper_chunk_docs[1]

Document(id='paper_chunk_1', metadata={'type': 'PaperChunk', 'mentions': ['http://w3id.org/midas-metadata/midas18', 'http://purl.obolibrary.org/obo/NCBITaxon_9606']}, page_content='discounted historical data for not-yet-observed time points; and coupled regions (when available). Our near-term weekly forecasts substantially over-estimated incidence early in the season when coupled models were not available. However, our forecast accuracy improved in absolute terms and relative to other teams once coupled solutions were available. In retrospective analysis, we found that the 2016-17 season was not typical: on average, coupled models performed better when fit without historically augmented data. Also, we tested a simple ensemble model for the 2016-17 season and found that it underperformed our subjective choice for all forecast targets. In this study, we were able to improve accuracy during a prospective forecasting exercise by coupling dynamics between regions. Although reduction of fore

NCBITAxon_9606 - is a not found? on purl?  Should be human

In [20]:
all_docs = ontology_docs + paper_chunk_docs

In [21]:
embedding_model = initialize_embedding_model()

In [22]:
analyze_chunk_similarities(all_docs, embedding_model)


Top similar PaperChunk pairs:
- paper_chunk_0  <->  paper_chunk_1  |  cos=0.631
  • Health planners use forecasts of key metrics associated with influenza-like illness (ILI); near-term weekly incidence, w...
  • discounted historical data for not-yet-observed time points; and coupled regions (when available). Our near-term weekly ...


two chunks. Not very useful. what's next?

What does the structure of a langchain document get used for?

In [23]:
vector_store = InMemoryVectorStore.from_documents(documents=all_docs, embedding=embedding_model)
retriever = setup_graph_retriever(vector_store)

In [24]:
results = retriever.invoke(query)

In [25]:
len(results)

10

go from here... What does it spit back?


In [26]:
direct_label_hits, synonym_hits = identify_matched_concepts(results, all_docs)

In [27]:
direct_label_hits

{'http://purl.obolibrary.org/obo/MONDO_0005812',
 'http://w3id.org/midas-metadata/midas111',
 'http://w3id.org/midas-metadata/midas18'}

In [28]:
synonym_hits

set()

In [29]:
all_docs[0]

Document(id='http://w3id.org/midas-metadata/midas6', metadata={'type': 'Class', 'label': 'modeling', 'parents': ['http://w3id.org/midas-metadata/midas2'], 'children': ['http://w3id.org/midas-metadata/midas16', 'http://w3id.org/midas-metadata/midas97', 'http://w3id.org/midas-metadata/midas68', 'http://w3id.org/midas-metadata/midas15'], 'equivalentClasses': [], 'domainOf': [], 'rangeOf': [], 'synonyms': []}, page_content='modeling')

In [30]:
retrieved_concepts = {doc.id for doc in results if doc.metadata.get("type") in ("Class", "Property")}
retrieved_concepts

{'http://purl.obolibrary.org/obo/MONDO_0005812',
 'http://purl.obolibrary.org/obo/MONDO_0024352',
 'http://purl.obolibrary.org/obo/NCBITaxon_10239',
 'http://purl.obolibrary.org/obo/NCBITaxon_11320',
 'http://purl.obolibrary.org/obo/NCBITaxon_11520',
 'http://purl.obolibrary.org/obo/NCIT_C35803',
 'http://w3id.org/midas-metadata/midas111',
 'http://w3id.org/midas-metadata/midas18',
 'http://w3id.org/midas-metadata/midas94'}

In [31]:
all_matched_concepts = retrieved_concepts.union(direct_label_hits).union(synonym_hits)

In [32]:
all_matched_concepts

{'http://purl.obolibrary.org/obo/MONDO_0005812',
 'http://purl.obolibrary.org/obo/MONDO_0024352',
 'http://purl.obolibrary.org/obo/NCBITaxon_10239',
 'http://purl.obolibrary.org/obo/NCBITaxon_11320',
 'http://purl.obolibrary.org/obo/NCBITaxon_11520',
 'http://purl.obolibrary.org/obo/NCIT_C35803',
 'http://w3id.org/midas-metadata/midas111',
 'http://w3id.org/midas-metadata/midas18',
 'http://w3id.org/midas-metadata/midas94'}

In [33]:
confidence_scores = calculate_confidence_scores(all_matched_concepts, query, all_docs, embedding_model, label_map)
confidence_scores

{'http://purl.obolibrary.org/obo/NCIT_C35803': 0.21638126499264862,
 'http://w3id.org/midas-metadata/midas18': 0.46193834575133436,
 'http://purl.obolibrary.org/obo/NCBITaxon_11520': 0.4868956458459191,
 'http://purl.obolibrary.org/obo/MONDO_0024352': 0.3250717009736466,
 'http://w3id.org/midas-metadata/midas94': 0.34108191638816887,
 'http://purl.obolibrary.org/obo/MONDO_0005812': 0.5797391739761515,
 'http://w3id.org/midas-metadata/midas111': 0.048513625071601764,
 'http://purl.obolibrary.org/obo/NCBITaxon_10239': 0.35712481715947314,
 'http://purl.obolibrary.org/obo/NCBITaxon_11320': 0.5310654772544301}

confidence scores based on vector dot product normalized, I think

In [34]:
results

[Document(id='paper_chunk_0', metadata={'_depth': 0, '_similarity_score': 0.7242454875481834, 'type': 'PaperChunk', 'mentions': ['http://w3id.org/midas-metadata/midas111', 'http://purl.obolibrary.org/obo/MONDO_0005812', 'http://w3id.org/midas-metadata/midas18']}, page_content='Health planners use forecasts of key metrics associated with influenza-like illness (ILI); near-term weekly incidence, week of season onset, week of peak, and intensity of peak. Here, we describe our participation in a weekly prospective ILI forecasting challenge for the United States for the 2016-17 season and subsequent evaluation of our performance. We implemented a metapopulation model framework with 32 model variants. Variants differed from each other in their assumptions about: the force-of-infection (FOI); use of uninformative priors; the use of discounted historical data for not-yet-observed time points; and the treatment of regions as either independent or coupled. Individual model variants were chosen s

In [35]:
matched_classes = [cid for cid in all_matched_concepts if G.nodes[cid]["type"] == "Class"]

In [36]:
matched_classes

['http://purl.obolibrary.org/obo/NCIT_C35803',
 'http://w3id.org/midas-metadata/midas18',
 'http://purl.obolibrary.org/obo/NCBITaxon_11520',
 'http://purl.obolibrary.org/obo/MONDO_0024352',
 'http://w3id.org/midas-metadata/midas94',
 'http://purl.obolibrary.org/obo/MONDO_0005812',
 'http://w3id.org/midas-metadata/midas111',
 'http://purl.obolibrary.org/obo/NCBITaxon_10239',
 'http://purl.obolibrary.org/obo/NCBITaxon_11320']

In [37]:
depth_map, roots = calculate_ontology_depth(G, label_map)

In [39]:
roots

['http://w3id.org/midas-metadata/midas03',
 'http://w3id.org/midas-metadata/midas10',
 'http://w3id.org/midas-metadata/midas2',
 'http://w3id.org/midas-metadata/midas77',
 'http://w3id.org/midas-metadata/midas5']

In [40]:
coverage_score, hierarchy_score, alignment_score, coherence_score, visited = calculate_relevance_scores(
        G, classes, matched_classes, depth_map
    )

# Calculate terminology scores
consistency_score, evidence_score, consistent_count = calculate_terminology_scores(
        matched_classes, direct_label_hits, synonym_hits
    )


total_score = sum([coverage_score, hierarchy_score, alignment_score,coherence_score, consistency_score, evidence_score])
total_score = min(total_score, 100.0)
verdict = generate_verdict(total_score)

print(f"**Executive Verdict:** **{verdict}** (Score: {total_score:.1f}/100)")


**Executive Verdict:** **Partially Relevant** (Score: 67.6/100)


In [41]:
top_concepts = sorted(all_matched_concepts, key=lambda cid: confidence_scores.get(cid, 0), reverse=True)

print_top_matches(top_concepts, G, label_map, confidence_scores, direct_label_hits, synonym_hits)
print_reasoned_paths(G, matched_classes, label_map, alignment_score)
print_gaps_and_next_steps(G, roots, matched_classes, visited, consistent_count, verdict, label_map)



**Top Matched Terms:**
| Paper Term | Ontology IRI | Ontology Label | Match Type | Confidence | Ontology Path |
|---|---|---|---|---|---|
| influenza | `http://purl.obolibrary.org/obo/MONDO_0005812` | influenza | label | 58% | influenza — subClassOf → viral respiratory tract infection |
| Influenza A virus | `http://purl.obolibrary.org/obo/NCBITaxon_11320` | Influenza A virus | embedding | 53% | Influenza A virus — subClassOf → Viruses |
| Influenza B virus | `http://purl.obolibrary.org/obo/NCBITaxon_11520` | Influenza B virus | embedding | 49% | Influenza B virus — subClassOf → Viruses |
| forecasting | `http://w3id.org/midas-metadata/midas18` | forecasting | label | 46% | forecasting — subClassOf → modeling purpose |
| Viruses | `http://purl.obolibrary.org/obo/NCBITaxon_10239` | Viruses | embedding | 36% | Viruses — subClassOf → pathogen |
| Vaccine-Preventable Disease | `http://w3id.org/midas-metadata/midas94` | Vaccine-Preventable Disease | embedding | 34% | Vaccine-Preventable Di